In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

In [ ]:
df = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df.shape , df_test.shape

((2000, 3074), (2000, 3073))

In [ ]:
X = df.drop(columns = ['id','target'], axis=1)
y = df['target']
X_test = df_test.drop('id', axis=1).copy()

X.shape , X_test.shape , y.shape

((2000, 3072), (2000, 3072), (2000,))

In [ ]:
X_train , X_val , y_train , y_val = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)

X_train.shape , X_val.shape , y_train.shape , y_val.shape

((1600, 3072), (400, 3072), (1600,), (400,))

In [ ]:
corr = X_train.corr().abs()

In [ ]:
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

In [ ]:
len(to_drop)

303

In [ ]:
X_train = X_train.drop(columns = to_drop, axis=1)
X_val = X_val.drop(columns = to_drop, axis=1)
X_test = X_test.drop(columns = to_drop, axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=1000)
X_train = selector.fit_transform(X_train, y_train)
X_val = selector.transform(X_val)
X_test = selector.transform(X_test)

In [ ]:
X_train, X_val, X_test = pd.DataFrame(X_train), pd.DataFrame(X_val), pd.DataFrame(X_test)

In [ ]:
X_train.shape , X_val.shape , X_test.shape

((1600, 1000), (400, 1000), (2000, 1000))

In [ ]:
model = LGBMClassifier(random_state=42)
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 160, number of negative: 1440
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026803 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 208853
[LightGBM] [Info] Number of data points in the train set: 1600, number of used features: 1000
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100000 -> initscore=-2.197225
[LightGBM] [Info] Start training from score -2.197225


LGBMClassifier(random_state=42)

In [ ]:
y_proba = model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

thresholds = np.arange(0.0, 1.01, 0.01)

best_threshold = 0
best_bal_acc = 0

for t in thresholds:
    y_pred = (y_proba > t).astype(int)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best balanced accuracy:", best_bal_acc)

Best threshold: 0.01
Best balanced accuracy: 0.6152777777777778


In [ ]:
y_pred = (y_proba > best_threshold).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score ,classification_report, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
print(classification_report(y_val, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_val, y_pred))
print("F1 score:", f1_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("ROC AUC score:", roc_auc_score(y_val, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.91      0.91       360
           1       0.28      0.33      0.30        40

    accuracy                           0.85       400
   macro avg       0.60      0.62      0.61       400
weighted avg       0.86      0.85      0.85       400

Balanced accuracy: 0.6152777777777778
F1 score: 0.2988505747126437
Precision: 0.2765957446808511
Recall: 0.325
ROC AUC score: 0.6152777777777777
Confusion matrix:
 [[326  34]
 [ 27  13]]


In [ ]:
#Hyper parameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

param_dist = {
    'n_estimators': [200, 350],
    'num_leaves': [31, 63],
    'learning_rate': [0.05, 0.1],
    'class_weight': ['balanced']
}
scorer = make_scorer(balanced_accuracy_score)
search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,
    scoring=scorer,
    cv=3,
    n_jobs=-1,
    verbose=1
)


search.fit(X_train, y_train)

print("Best Params:", search.best_params_)
print("Best Balanced Accuracy:", search.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[LightGBM] [Info] Number of positive: 160, number of negative: 1440
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029817 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 208853
[LightGBM] [Info] Number of data points in the train set: 1600, number of used features: 1000
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Best Params: {'num_leaves': 31, 'n_estimators': 200, 'learning_rate': 0.05, 'class_weight': 'balanced'}
Best Balanced Accuracy: 0.5280886035406476


In [ ]:
best_model = search.best_estimator_

y_proba = best_model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

thresholds = np.arange(0.0, 1.01, 0.01)

best_threshold = 0
best_bal_acc = 0

for t in thresholds:
    y_pred = (y_proba > t).astype(int)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best balanced accuracy:", best_bal_acc)

Best threshold: 0.01
Best balanced accuracy: 0.6513888888888889


In [ ]:
y_pred = (y_proba > best_threshold).astype(int)

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
print(classification_report(y_val, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_val, y_pred))
print("F1 score:", f1_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("ROC AUC score:", roc_auc_score(y_val, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))
print('Accuracy :', accuracy_score(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.80      0.86       360
           1       0.22      0.50      0.31        40

    accuracy                           0.77       400
   macro avg       0.58      0.65      0.58       400
weighted avg       0.86      0.77      0.81       400

Balanced accuracy: 0.6513888888888889
F1 score: 0.3053435114503817
Precision: 0.21978021978021978
Recall: 0.5
ROC AUC score: 0.6513888888888889
Confusion matrix:
 [[289  71]
 [ 20  20]]
Accuracy : 0.7725


In [ ]:
y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba > best_threshold).astype(int)


In [ ]:
pd.DataFrame({'id': df_test['id'], 'target': y_test_pred}).to_csv('submission_lgbm_HP_k1000.csv', index=False)